# Лабораторная работа №2
## Обработка пропусков в данных, кодирование категориальных признаков, масштабирование данных.

**Выполнил:** Зазовский В.Д.  
**Группа:** ИБМ3-64Б  
**Дата:** 2026

### Цель работы
Изучение методов обработки пропусков, кодирования категориальных признаков и масштабирования данных.

### Описание набора данных
Используется набор данных **Titanic** (часть) — данные о пассажирах Титаника с пропусками и категориальными признаками.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Создадим датасет с пропусками на основе известных данных
np.random.seed(42)

# Используем make_classification для создания репрезентативного набора
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=200, n_features=4, n_informative=3, n_redundant=1,
                            n_classes=2, random_state=42)

df = pd.DataFrame(X, columns=['age', 'income', 'score', 'experience'])
df['target'] = y

# Добавим категориальный признак
df['category'] = pd.cut(df['score'], bins=3, labels=['low', 'medium', 'high'])

# Создадим пропуски искусственно (10-15%)
for col in ['age', 'income']:
    missing_idx = np.random.choice(df.index, size=int(0.12*len(df)), replace=False)
    df.loc[missing_idx, col] = np.nan

# Пропуски в категориальном признаке
missing_cat = np.random.choice(df.index, size=int(0.08*len(df)), replace=False)
df.loc[missing_cat, 'category'] = np.nan

print("=== ИСХОДНЫЙ НАБОР ДАННЫХ ===")
print(df.head(10))
print("\nРазмер:", df.shape)
print("\nТипы данных:")
print(df.dtypes)
print("\n=== ПРОПУСКИ ===")
print(df.isnull().sum())
print(f"\nПроцент пропусков:")
print((df.isnull().sum() / len(df) * 100).round(2))


In [ ]:
# Визуализация пропусков
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis')
plt.title('Визуализация пропусков в данных', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/mnt/agents/output/tmo_labs/lab2/missing_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# === МЕТОД 1: Удаление строк с пропусками ===
df_drop = df.dropna()
print(f"Исходный размер: {df.shape[0]} строк")
print(f"После удаления: {df_drop.shape[0]} строк")
print(f"Удалено: {df.shape[0] - df_drop.shape[0]} строк ({((df.shape[0]-df_drop.shape[0])/df.shape[0]*100):.1f}%)")


In [ ]:
# === МЕТОД 2: Заполнение пропусков ===
df_filled = df.copy()

# Заполнение числовых признаков медианой
num_imputer = SimpleImputer(strategy='median')
df_filled[['age', 'income']] = num_imputer.fit_transform(df_filled[['age', 'income']])

# Заполнение категориального признака модой
cat_imputer = SimpleImputer(strategy='most_frequent')
df_filled['category'] = cat_imputer.fit_transform(df_filled[['category']]).ravel()

print("Пропуски после заполнения:")
print(df_filled.isnull().sum())
print("\nСтатистика числовых признаков после заполнения:")
print(df_filled[['age', 'income']].describe().round(3))


In [ ]:
# === КОДИРОВАНИЕ КАТЕГОРИАЛЬНЫХ ПРИЗНАКОВ ===

# Метод 1: Label Encoding
le = LabelEncoder()
df_filled['category_le'] = le.fit_transform(df_filled['category'])
print("Label Encoding:")
print(dict(zip(le.classes_, le.transform(le.classes_))))

# Метод 2: One-Hot Encoding
df_encoded = pd.get_dummies(df_filled, columns=['category'], prefix='cat')
print("\nOne-Hot Encoding создал столбцы:")
print([c for c in df_encoded.columns if c.startswith('cat_')])

print("\nПервые 5 строк после кодирования:")
print(df_encoded[['category_le', 'cat_high', 'cat_low', 'cat_medium']].head())


In [ ]:
# === МАСШТАБИРОВАНИЕ ДАННЫХ ===
features_to_scale = ['age', 'income', 'score', 'experience']

# Метод 1: StandardScaler (Z-нормализация)
scaler_std = StandardScaler()
df_encoded[['age_std', 'income_std', 'score_std', 'exp_std']] = scaler_std.fit_transform(df_encoded[features_to_scale])

# Метод 2: MinMaxScaler
scaler_minmax = MinMaxScaler()
df_encoded[['age_mm', 'income_mm', 'score_mm', 'exp_mm']] = scaler_minmax.fit_transform(df_encoded[features_to_scale])

print("=== СРАВНЕНИЕ МАСШТАБИРОВАНИЯ ===")
print("\nStandardScaler (среднее≈0, std≈1):")
print(df_encoded[['age_std', 'income_std', 'score_std', 'exp_std']].describe().round(3))
print("\nMinMaxScaler (диапазон [0,1]):")
print(df_encoded[['age_mm', 'income_mm', 'score_mm', 'exp_mm']].describe().round(3))


In [ ]:
# Визуализация масштабирования
fig, axes = plt.subplots(3, 1, figsize=(12, 12))

# Исходные данные
axes[0].boxplot([df_filled['age'].dropna(), df_filled['income'].dropna(), 
                 df_filled['score'], df_filled['experience']])
axes[0].set_title('Исходные данные', fontsize=12, fontweight='bold')
axes[0].set_xticklabels(['age', 'income', 'score', 'experience'])
axes[0].grid(True, alpha=0.3)

# StandardScaler
axes[1].boxplot([df_encoded['age_std'], df_encoded['income_std'], 
                 df_encoded['score_std'], df_encoded['exp_std']])
axes[1].set_title('StandardScaler (Z-нормализация)', fontsize=12, fontweight='bold')
axes[1].set_xticklabels(['age', 'income', 'score', 'experience'])
axes[1].grid(True, alpha=0.3)

# MinMaxScaler
axes[2].boxplot([df_encoded['age_mm'], df_encoded['income_mm'], 
                 df_encoded['score_mm'], df_encoded['exp_mm']])
axes[2].set_title('MinMaxScaler ([0, 1])', fontsize=12, fontweight='bold')
axes[2].set_xticklabels(['age', 'income', 'score', 'experience'])
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/mnt/agents/output/tmo_labs/lab2/scaling_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Итоговый набор данных
print("=== ИТОГОВЫЙ НАБОР ДАННЫХ ===")
print(f"Размер: {df_encoded.shape}")
print(f"Пропусков: {df_encoded.isnull().sum().sum()}")
print("\nСтолбцы:", list(df_encoded.columns))
print("\nПервые 5 строк:")
print(df_encoded.head())


### Выводы
1. Пропуски заполнены медианой (числовые) и модой (категориальные) — сохранено 100% данных.
2. Label Encoding преобразовал категории в числа 0/1/2.
3. One-Hot Encoding создал бинарные признаки без введения порядка.
4. StandardScaler центрировал данные (среднее≈0), MinMaxScaler привёл к диапазону [0,1].
5. Выбор метода масштабирования зависит от алгоритма: KNN/SVM — StandardScaler, нейросети — MinMaxScaler.
